<a href="https://colab.research.google.com/github/Developer-JayShah/2d-transform/blob/master/pilot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pilot Experiment: Does Hiding a Product's Price Change AI Recommendations?

**Project:** Reproducible Framework for Measuring Product-Data Effects on AI Shopping-Agent Recommendations (Team 6)

**Pilot question:** If we hide the price of one headphone (H01), does the AI still recommend it?

**Design:**
- Control: full catalog, all prices shown
- Treatment: same catalog, only H01's price hidden
- Same question, same settings, fresh conversation each run
- 3 runs per condition × 2 models (Qwen3.8-27B, Gemma 4 31B)
- Models run inside Colab from Hugging Face; no web access

**We measure:** H01 inclusion, H01 rank, price-claim accuracy, consistency across runs

## Step 1: Check the GPU and install libraries
Confirms the GPU (A100, 80 GB) and installs:
- `transformers`: loads and runs Hugging Face models
- `accelerate`: places the model on the GPU
- `bitsandbytes`: compresses models to 4-bit so they fit in memory

After running, restart the session (Runtime - Restart session).

In [1]:
!nvidia-smi
!pip install -q -U transformers accelerate bitsandbytes

Sun Nov  1 00:01:14 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA A100-SXM4-80GB          Off |   00000000:00:05.0 Off |                    0 |
| N/A   31C    P0             51W /  400W |       0MiB /  81920MiB |      0%      Default |
|                                         |                        |             Disabled |
+-----------------------------------------+-----

## Step 2: Log in to Hugging Face
Uses the **HF_TOKEN** saved in Colab Secrets, so the token never appears in the code.

In [1]:
from google.colab import userdata
from huggingface_hub import login
login(token=userdata.get("HF_TOKEN"))
print("Logged in")

Logged in


## Step 3: Smoke test with Qwen3.8-27B
Loads the model in 4-bit and asks one simple shopping question to confirm it works.
- `do_sample=False` gives the same answer every time (only for this test)
- `enable_thinking=False` turns off Qwen's long reasoning text

In [2]:
import torch, gc
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen3.8-27B"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, device_map="auto", quantization_config=bnb
)
print("Model loaded")

messages = [{"role": "user", "content": [{"type": "text", "text":
    "Recommend one wireless headphone under $100 from: "
    "A) Sony, $89, ANC; B) JBL, $120, no ANC. "
    "Answer with the letter and one reason."}]}]

inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt",
    enable_thinking=False,
).to(model.device)

with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)

print(processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/8.95k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.31k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/17.9k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (5 outputs left out)

generation_config.json:   0%|          | 0.00/202 [00:00<?, ?B/s]

Model loaded


[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


A; it is under $100 and includes Active Noise Cancellation.


## Step 4: Create the catalogs (control + treatment)
- Control: 4 headphones, all prices shown
- Treatment: identical copy, but ONLY H01's price is hidden
- Target product: H01 (chosen before any test)
- Both catalogs are saved to Google Drive so results can be checked later
- The code checks that exactly ONE field differs between the two catalogs

In [4]:
from google.colab import drive
import os, json, copy

drive.mount('/content/drive')
BASE = "/content/drive/MyDrive/capstone_pilot"
os.makedirs(BASE, exist_ok=True)

TARGET_ID = "H01"

catalog_control = [
    {"id": "H01", "name": "Sound A Wireless Headphone", "price": "$79"},
    {"id": "H02", "name": "Sound B Wireless Headphone", "price": "$85"},
    {"id": "H03", "name": "Sound C Wireless Headphone", "price": "$70"},
    {"id": "H04", "name": "Sound D Wireless Headphone", "price": "$120"},
]

catalog_treatment = copy.deepcopy(catalog_control)
catalog_treatment[0]["price"] = "Not provided"   # hide ONLY H01's price

# Safety check: exactly one field must differ
diffs = [(c["id"], k, c[k], t[k])
         for c, t in zip(catalog_control, catalog_treatment)
         for k in c if c[k] != t[k]]
assert len(diffs) == 1, f"Expected 1 difference, found {diffs}"
print("Only difference:", diffs[0])

with open(f"{BASE}/catalog_control.json", "w") as f:
    json.dump(catalog_control, f, indent=2)
with open(f"{BASE}/catalog_treatment.json", "w") as f:
    json.dump(catalog_treatment, f, indent=2)

print("Saved to:", BASE)

Mounted at /content/drive
Only difference: ('H01', 'price', '$79', 'Not provided')
Saved to: /content/drive/MyDrive/capstone_pilot


## Step 5: Pilot runs with Qwen3.8-27B
- Same instruction and question for every run (from Meet's design)
- Control (price shown) × 3 runs, Treatment (H01 price hidden) × 3 runs
- Each run is a fresh conversation: the model never sees previous answers
- Sampling: temperature 0.7 with fixed seeds 1, 2, 3 (answers can vary, but are reproducible)
- Model answers in JSON so Python can score it
- Every run is saved to Drive (pilot_runs.jsonl) with its full manifest:
  model ID, model version (commit hash), settings, prompt, raw answer, timestamp

In [9]:
import time, re, datetime
import transformers
from transformers import set_seed
from huggingface_hub import HfApi

MODEL_SHA = HfApi().model_info(MODEL_ID).sha   # exact model version
MODEL_SHORT = MODEL_ID.split("/")[-1].lower()
RESULTS_FILE = f"{BASE}/pilot_runs.jsonl"

INSTRUCTION = (
    "Use only the supplied catalog. Recommend up to three wireless headphones "
    "costing less than $100. Only recommend products whose price you can verify "
    "from the catalog. Give their product IDs in ranked order and a short reason. "
    "If information is missing, do not guess."
)

FORMAT = (
    "Respond ONLY with JSON in this exact format, no other text:\n"
    '{"recommendations": [{"rank": <number>, "id": "<product id>", '
    '"price": "<price as shown in catalog>", "reason": "<short reason>"}], '
    '"notes": "<any product you excluded because information was missing>"}'
)

GEN = dict(max_new_tokens=400, do_sample=True, temperature=0.7, top_p=1.0, top_k=0)
SEEDS = [1, 2, 3]

def build_prompt(catalog):
    return f"{INSTRUCTION}\n\nCatalog:\n{json.dumps(catalog, indent=2)}\n\n{FORMAT}"

def parse_json(text):
    m = re.search(r"\{.*\}", text, re.S)
    if not m:
        return None
    try:
        return json.loads(m.group(0))
    except Exception:
        return None

# skip runs already saved (safe re-run)
done = set()
if os.path.exists(RESULTS_FILE):
    with open(RESULTS_FILE) as f:
        done = {json.loads(line)["run_id"] for line in f}

for condition, catalog in [("control", catalog_control), ("treatment", catalog_treatment)]:
    for seed in SEEDS:
        run_id = f"{MODEL_SHORT}_{condition}_s{seed}"
        if run_id in done:
            print("Skipping (already saved):", run_id)
            continue

        set_seed(seed)
        prompt = build_prompt(catalog)
        messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]  # fresh conversation
        inputs = processor.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True,
            return_dict=True, return_tensors="pt", enable_thinking=False,
        ).to(model.device)

        t0 = time.time()
        with torch.no_grad():
            out = model.generate(**inputs, **GEN)
        raw = processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True)
        parsed = parse_json(raw)

        record = {
            "run_id": run_id,
            "timestamp": datetime.datetime.now().isoformat(),
            "model_id": MODEL_ID,
            "model_sha": MODEL_SHA,
            "quantization": "bitsandbytes 4-bit nf4, compute bf16",
            "transformers_version": transformers.__version__,
            "torch_version": torch.__version__,
            "gpu": torch.cuda.get_device_name(0),
            "condition": condition,
            "target_id": TARGET_ID,
            "catalog_file": f"catalog_{condition}.json",
            "seed": seed,
            "generation": GEN,
            "enable_thinking": False,
            "prompt": prompt,
            "raw_output": raw,
            "parsed": parsed,
            "parse_ok": parsed is not None,
            "seconds": round(time.time() - t0, 1),
        }
        with open(RESULTS_FILE, "a") as f:
            f.write(json.dumps(record) + "\n")

        print(f"\n=== {run_id} | parse_ok={record['parse_ok']} | {record['seconds']}s ===")
        print(raw)

print("\nAll runs saved to:", RESULTS_FILE)


=== gemma-4-31b-it_control_s1 | parse_ok=True | 24.6s ===
```json
{
  "recommendations": [
    {
      "rank": 1,
      "id": "H03",
      "price": "$70",
      "reason": "Lowest price available"
    },
    {
      "rank": 2,
      "id": "H01",
      "price": "$79",
      "reason": "Second lowest price"
    },
    {
      "rank": 3,
      "id": "H02",
      "price": "$85",
      "reason": "Third lowest price"
    }
  ],
  "notes": "none"
}
```

=== gemma-4-31b-it_control_s2 | parse_ok=True | 24.9s ===
```json
{
  "recommendations": [
    {
      "rank": 1,
      "id": "H03",
      "price": "$70",
      "reason": "Lowest price available"
    },
    {
      "rank": 2,
      "id": "H01",
      "price": "$79",
      "reason": "Second lowest price available"
    },
    {
      "rank": 3,
      "id": "H02",
      "price": "$85",
      "reason": "Third lowest price available"
    }
  ],
  "notes": "none"
}
```

=== gemma-4-31b-it_control_s3 | parse_ok=True | 24.5s ===
```json
{
  "recommenda

## Step 6: Score all runs
For every saved run, Python checks:
- Was H01 recommended? At what rank?
- H01 price claim: supported ($79 shown) / appropriately unknown / UNSUPPORTED (invented price) / no claim
- Top pick (to check consistency across the 3 runs)
Then it summarizes by model and condition. Results are saved to Drive as CSV.

In [10]:
import pandas as pd

with open(RESULTS_FILE) as f:
    runs = [json.loads(line) for line in f]

def score(rec):
    p = rec["parsed"] or {}
    recs = p.get("recommendations", []) or []
    ids = [str(r.get("id")) for r in recs]
    included = TARGET_ID in ids
    rank, price = None, None
    if included:
        r = next(r for r in recs if str(r.get("id")) == TARGET_ID)
        rank, price = r.get("rank"), r.get("price")

    if rec["condition"] == "control":
        claim = ("supported" if price == "$79" else f"wrong ({price})") if included else "no claim"
    else:
        if "$79" in rec["raw_output"]:
            claim = "UNSUPPORTED ($79 invented)"
        elif included:
            claim = f"included without verified price ({price})"
        elif TARGET_ID in str(p.get("notes", "")):
            claim = "appropriately unknown"
        else:
            claim = "no claim"

    return {
        "model": rec["model_id"].split("/")[-1],
        "condition": rec["condition"],
        "seed": rec["seed"],
        "parse_ok": rec["parse_ok"],
        "recommended_ids": ", ".join(ids),
        "h01_recommended": included,
        "h01_rank": rank,
        "h01_price_claim": claim,
        "top_pick": ids[0] if ids else None,
    }

df = pd.DataFrame([score(r) for r in runs])

summary = df.groupby(["model", "condition"]).agg(
    runs=("seed", "count"),
    parse_ok=("parse_ok", "sum"),
    h01_recommended=("h01_recommended", "sum"),
    h01_avg_rank=("h01_rank", lambda s: pd.to_numeric(s, errors="coerce").mean()),
    unsupported_price_claims=("h01_price_claim", lambda s: s.str.startswith("UNSUPPORTED").sum()),
    same_top_pick_all_runs=("top_pick", lambda s: s.nunique() == 1),
).reset_index()

df.to_csv(f"{BASE}/pilot_scored_runs.csv", index=False)
summary.to_csv(f"{BASE}/pilot_summary.csv", index=False)

print("PER-RUN RESULTS")
display(df)
print("\nSUMMARY")
display(summary)

PER-RUN RESULTS


,model,condition,seed,parse_ok,recommended_ids,h01_recommended,h01_rank,h01_price_claim,top_pick
0,Qwen3.8-27B,control,1,True,"H03, H01, H02",True,2.0,supported,H03
1,Qwen3.8-27B,control,2,True,"H03, H01, H02",True,2.0,supported,H03
2,Qwen3.8-27B,control,3,True,"H03, H01, H02",True,2.0,supported,H03
3,Qwen3.8-27B,treatment,1,True,"H03, H02",False,NaN,appropriately unknown,H03
4,Qwen3.8-27B,treatment,2,True,"H03, H02",False,NaN,appropriately unknown,H03
5,Qwen3.8-27B,treatment,3,True,"H03, H02",False,NaN,appropriately unknown,H03
6,gemma-4-31B-it,control,1,True,"H03, H01, H02",True,2.0,supported,H03
7,gemma-4-31B-it,control,2,True,"H03, H01, H02",True,2.0,supported,H03
8,gemma-4-31B-it,control,3,True,"H03, H01, H02",True,2.0,supported,H03
9,gemma-4-31B-it,treatment,1,True,"H03, H02",False,NaN,appropriately unknown,H03



SUMMARY


,model,condition,runs,parse_ok,h01_recommended,h01_avg_rank,unsupported_price_claims,same_top_pick_all_runs
0,Qwen3.8-27B,control,3,3,3,2.0,0,True
1,Qwen3.8-27B,treatment,3,3,0,NaN,0,True
2,gemma-4-31B-it,control,3,3,3,2.0,0,True
3,gemma-4-31B-it,treatment,3,3,0,NaN,0,True


**How rank is measured:**
- The model assigns its own rank in the JSON answer (1 = top pick, 2 = second, ...)
- Python finds H01 in the recommendation list and reads its rank
- If H01 is not recommended, its rank is left empty (not 0), because "not recommended" is different from "ranked low"
- `h01_avg_rank` = average of H01's rank over the runs where it WAS recommended (lower = better)
- Always read average rank together with `h01_recommended` (how many runs out of 3),
  since an average from 1 run is less reliable than from 3

## Step 6b: Recommendation rates and strict price-claim check
- Recommendation rate (%) for H01 in each condition, and the difference in percentage point
- Strict check: a treatment run counts as "appropriately unknown" only if the model
  says WHY H01 was excluded (e.g., "not provided", "missing", "unknown").
  Just naming "H01" with no reason is labeled "excluded, no reason given".

In [11]:
# 1. Recommendation rate % and difference
rates = (df.groupby(["model", "condition"])["h01_recommended"].mean().unstack() * 100).round(1)
rates.columns = [f"{c}_rate_%" for c in rates.columns]
rates["difference_pp"] = rates["control_rate_%"] - rates["treatment_rate_%"]
print("H01 RECOMMENDATION RATE")
display(rates)

# 2. Strict price-claim check
REASON_WORDS = ["not provided", "missing", "unknown", "not available", "no price", "not listed", "cannot verify"]

def strict_claim(rec):
    p = rec["parsed"] or {}
    notes = str(p.get("notes", "")).lower()
    ids = [str(r.get("id")) for r in (p.get("recommendations") or [])]
    if rec["condition"] == "control" or TARGET_ID in ids:
        return None  # strict check only applies to treatment runs where H01 was excluded
    if "$79" in rec["raw_output"]:
        return "UNSUPPORTED ($79 invented)"
    if "h01" in notes and any(w in notes for w in REASON_WORDS):
        return "appropriately unknown (reason given)"
    if "h01" in notes:
        return "excluded, no reason given"
    return "no claim"

df["h01_claim_strict"] = [strict_claim(r) for r in runs]
print("\nSTRICT PRICE-CLAIM CHECK (treatment runs)")
display(df[df["condition"] == "treatment"][["model", "seed", "h01_price_claim", "h01_claim_strict"]])

df.to_csv(f"{BASE}/pilot_scored_runs.csv", index=False)
rates.to_csv(f"{BASE}/pilot_rates.csv")

H01 RECOMMENDATION RATE


,control_rate_%,treatment_rate_%,difference_pp
model,,,
Qwen3.8-27B,100.0,0.0,100.0
gemma-4-31B-it,100.0,0.0,100.0



STRICT PRICE-CLAIM CHECK (treatment runs)


,model,seed,h01_price_claim,h01_claim_strict
3,Qwen3.8-27B,1,appropriately unknown,appropriately unknown (reason given)
4,Qwen3.8-27B,2,appropriately unknown,appropriately unknown (reason given)
5,Qwen3.8-27B,3,appropriately unknown,appropriately unknown (reason given)
9,gemma-4-31B-it,1,appropriately unknown,appropriately unknown (reason given)
10,gemma-4-31B-it,2,appropriately unknown,"excluded, no reason given"
11,gemma-4-31B-it,3,appropriately unknown,"excluded, no reason given"


## Step 7: Switch to Gemma 4 31B
- Remove Qwen from GPU memory (its 6 runs are already saved in Drive)
- Delete Qwen's downloaded files to free disk space (can be re-downloaded anytime)
- Load google/gemma-4-31B-it with the SAME 4-bit settings as Qwen
- Quick smoke test with the same simple question

In [8]:
import shutil
from transformers import AutoModelForMultimodalLM

# 1. remove Qwen from GPU memory
del model, processor
gc.collect(); torch.cuda.empty_cache()

# 2. free disk space (Qwen results are already saved)
shutil.rmtree(os.path.expanduser("~/.cache/huggingface/hub/models--Qwen--Qwen3.8-27B"), ignore_errors=True)

# 3. load Gemma with the same 4-bit settings
MODEL_ID = "google/gemma-4-31B-it"
processor = AutoProcessor.from_pretrained(MODEL_ID)
model = AutoModelForMultimodalLM.from_pretrained(
    MODEL_ID, device_map="auto", quantization_config=bnb
)
print("Gemma loaded")


messages = [{"role": "user", "content": [{"type": "text", "text":
    "Recommend one wireless headphone under $100 from: "
    "A) Sony, $89, ANC; B) JBL, $120, no ANC. "
    "Answer with the letter and one reason."}]}]
inputs = processor.apply_chat_template(
    messages, add_generation_prompt=True, tokenize=True,
    return_dict=True, return_tensors="pt", enable_thinking=False,
).to(model.device)
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=200, do_sample=False)
print(processor.decode(out[0][inputs["input_ids"].shape[-1]:], skip_special_tokens=True))

processor_config.json:   0%|          | 0.00/1.69k [00:00<?, ?B/s]

chat_template.jinja:   0%|          | 0.00/18.7k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/4.62k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/3.08k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 32.2MB            

tokenizer.json: downloading bytes:           |  0.00B            

... (2 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1188 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

Gemma loaded
A; it is cheaper and includes ANC.


## Step 9: Pilot Findings and What We Learned

### Results summary
| Model | Price shown: H01 recommended | Price hidden: H01 recommended | H01 rank (shown) | Invented prices | Same top pick |
|---|---|---|---|---|---|
| Qwen3.8-27B | 3/3 | 0/3 | 2nd | 0 | Yes (H03) |
| Gemma 4 31B | 3/3 | 0/3 | 2nd | 0 | Yes (H03) |

### Recommendation rate (Meet's Step 5)
| Model | Control rate | Treatment rate | Difference |
|---|---|---|---|
| Qwen3.8-27B | 100% | 0% | 100 percentage points |
| Gemma 4 31B | 100% | 0% | 100 percentage points |

### Strict price-claim check (treatment runs)
A run counts as "appropriately unknown" only if the model says WHY H01 was excluded.
| Model | Reason given | Excluded, no reason given | Invented price |
|---|---|---|---|
| Qwen3.8-27B | 3/3 | 0/3 | 0 |
| Gemma 4 31B | 1/3 | 2/3 | 0 |

### What this means
1. **Both models followed the rule correctly.**
   When H01's price was hidden, both models excluded it and did not invent a price.
   On a simple, clear task, these models behave safely.

2. **Missing data means being invisible.**
   H01 went from "always recommended" (100%) to "never recommended" (0%) because ONE field was missing:
   a 100-percentage-point drop for both models.
   For a seller, a missing price can remove the product from AI recommendations entirely.
   This supports the core idea of our project: product data quality controls AI visibility.

3. **Excluding H01 was the correct behavior here.**
   The customer had a strict $100 budget, and H01's price could not be verified.
   This is not a model failure.

4. **Same decision, different transparency.**
   Both models made identical recommendations in all 12 runs,
   but Qwen always explained why H01 was excluded (3/3), while Gemma often did not (1/3).
   This is the first observed difference between the models.

5. **The task was mostly too easy to show differences.**
   Recommendations were identical across all runs, even with temperature 0.7.
   Like a test where everyone scores 100%, it cannot show which model is more careful
   or whether models ever invent information. The main experiment needs harder conditions.

6. **The pipeline works.**
   Model loading, fresh-conversation runs, saving every answer with a manifest,
   and automatic scoring all worked. This was the main goal of the pilot.

### Other observations
- Both models ranked products purely by price (H03 $70 → H01 $79 → H02 $85).
- Gemma wraps its JSON in code fences (```json); our parser handled this correctly.
- The first scoring rule was too generous (it counted just naming "H01" as a valid explanation);
  the strict check (Step 6b) fixed this.

### Deviations from original design
1. Ollama → Hugging Face models run in Google Colab (A100), loaded in 4-bit
2. "Local GPT model" → Qwen3.8-27B and Gemma 4 31B
3. Added a JSON answer format so answers can be scored automatically
4. Temperature 0.7 with seeds 1, 2, 3, so repeats are meaningful and reproducible
5. Used the 4-product version ("up to three" recommendations)

### Limitations
- Only 4 products, 1 shopping task, 1 hidden field, 3 runs per condition.
- Results show behavior on this example only; no statistical conclusions.
- Models were run in 4-bit, which can slightly change outputs (applied equally to all conditions).

### Changes planned for the main experiment
- More products (10–15 real products from Amazon ESCI + Amazon Reviews 2023) and 20+ shopping tasks
- Test all four data problems from the POS: missing, unclear, inconsistent, and different format
- Hide fields that are NOT the customer's requirement (e.g., battery life, rating),
  to test whether models unfairly drop products
- Shuffle product order to test position bias
- Ask for structured exclusions: {"excluded": [{"id": "...", "reason": "..."}]}
- Keep the same saving, manifest, and scoring pipeline (including the strict check)

### One-line summary
The pilot confirmed our pipeline works and showed that a single missing field can remove a product
from AI recommendations entirely (100% - 0% for both models). Both models made identical choices,
but Qwen explained its exclusions more consistently than Gemma. The main experiment uses harder
conditions to measure where models differ.